In [ ]:
import os 
import pandas as pd 
import numpy as np

from dotenv import load_dotenv

import requests 
import json 

from bs4 import BeautifulSoup
import io

import csv

import time

import geopandas as gpd

In [ ]:
#### setting dictionaries - mappings 

In [73]:
import os
proj_root = os.path.abspath(os.path.join(os.getcwd(), ".."))   # notebooks/ -> project root
data_dir = os.path.join(proj_root, "data", "raw")
os.makedirs(data_dir, exist_ok=True)

In [74]:
import os, sys
print(data_dir)
print("cwd:", os.getcwd())
print("src exists:", os.path.exists(os.path.join(os.getcwd(), "src")))
print("sys.path[0]:", sys.path[0])

c:\Users\marki\Data-Processing-in-Python---Project\data\raw
cwd: c:\Users\marki\Data-Processing-in-Python---Project\notebooks
src exists: False
sys.path[0]: c:\Users\marki\AppData\Local\Programs\Python\Python313\python313.zip


In [75]:
### chmi weather stations - data processing

df_chmi_stat = pd.read_csv(os.path.join(data_dir, "chmi_weather_stations_metadata.csv"))

df_chmi_stat = df_chmi_stat[df_chmi_stat['FULL_NAME'].str.contains('Praha', case=False, na=False)]

wsi_to_drop = [
    '0-203-0-11201020001', #Praha, Vinohrady - Flora	
    '0-203-0-11202007001', #Praha, Suchdol
    '0-203-0-11105048001', #Praha, Zadní Kopanina
    '0-203-0-11201020003' #Praha, Chodov
    ]

df_chmi_stat = df_chmi_stat[~df_chmi_stat['WSI'].isin(wsi_to_drop)]

df_chmi_stat["END_DATE_DT"] = pd.to_datetime(df_chmi_stat["END_DATE"], utc=True, errors="coerce")

df_chmi_stat = (
    df_chmi_stat.sort_values("END_DATE_DT")
    .drop_duplicates(subset="WSI", keep="last")
)

now_utc = pd.Timestamp.now(tz="UTC")
df_chmi_stat = df_chmi_stat[
    (df_chmi_stat["END_DATE_DT"] >= now_utc)
]

In [76]:
df_chmi_stat

,WSI,GH_ID,BEGIN_DATE,END_DATE,FULL_NAME,GEOGR1,GEOGR2,ELEVATION,END_DATE_DT
1101,0-203-0-10904013001,P1PKOM01,2017-12-07T12:00:00Z,3999-12-31T23:59:00Z,"Praha, Komořany",14.406360,49.988600,213.00,3999-12-31 23:59:00+00:00
75,0-20000-0-11567,P1PKBE01,2010-10-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Kbely",14.538056,50.123333,284.50,3999-12-31 23:59:00+00:00
1429,0-203-0-11201024001,P1PBRE01,2013-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Břevnov",14.352689,50.080843,355.00,3999-12-31 23:59:00+00:00
68,0-20000-0-11520,P1PLIB01,2018-05-17T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Libuš",14.446944,50.007778,302.04,3999-12-31 23:59:00+00:00
65,0-20000-0-11519,P1PKAR01,2024-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Karlov",14.427778,50.069167,260.50,3999-12-31 23:59:00+00:00
2214,0-203-0-11514,P1PKLE01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416923,50.086634,190.70,3999-12-31 23:59:00+00:00
62,0-20000-0-11518,P1PRUZ01,2000-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Ruzyně",14.255556,50.100278,364.00,3999-12-31 23:59:00+00:00
2217,0-203-0-11515,P1PKLM01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416436,50.086341,190.70,3999-12-31 23:59:00+00:00
1273,0-203-0-11101007001,L4PRAH01,2015-03-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Brdy",13.818380,49.658890,862.00,3999-12-31 23:59:00+00:00


In [77]:
### discionary of wsi codes

wsi_dict = dict(
    zip(
        df_chmi_stat["WSI"].astype(str),
        df_chmi_stat["FULL_NAME"].astype(str)  
    )
)

with open(os.path.join(data_dir, "wsi_dict.csv"),"w",encoding="utf-8-sig",newline="") as f:
    writer=csv.writer(f); writer.writerow(["key","value"]); writer.writerows(wsi_dict.items())

In [78]:
### chmi weather variables

### filtering only needed ones

df_chmi_vars = pd.read_csv(os.path.join(data_dir, "chmi_weather_variables_metadata.csv"))

df_chmi_vars = df_chmi_vars[df_chmi_vars['WSI'].astype(str).isin(wsi_dict)]

### dictionary of variables abbreviations and names
chmi_vars_dict = dict(
    zip(
        df_chmi_vars['EG_EL_ABBREVIATION'].astype(str),
        df_chmi_vars['NAME'].astype(str)  
    )
)

with open(os.path.join(data_dir, "chmi_vars_dict.csv"),"w",encoding="utf-8-sig",newline="") as f:
    writer=csv.writer(f); writer.writerow(["key","value"]); writer.writerows(chmi_vars_dict.items())

In [ ]:
### chmi weather 10min data

df_weather_10min = pd.read_csv(os.path.join(data_dir, "weather_data_10min.csv"))

# it gives flag warning, but i assume flag is not of interest


C:\Users\marki\AppData\Local\Temp\ipykernel_16988\1676926942.py:3: DtypeWarning: Columns (0: FLAG) have mixed types. Specify dtype option on import or set low_memory=False.
  df_weather = pd.read_csv(os.path.join(data_dir, "weather_data_10min.csv"))


In [88]:
### chmi weather 1h data

df_weather = pd.read_csv(os.path.join(data_dir, "weather_data_1hour.csv"))



In [89]:
df_weather.head()

,STATION,ELEMENT,DT,VAL,FLAG,QUALITY,WSI,YEAR,MONTH
0,0-203-0-10904013001,E,2025-11-01T00:00:00Z,8.0,NaN,0.0,0-203-0-10904013001,2025,11
1,0-203-0-10904013001,E,2025-11-01T01:00:00Z,7.7,NaN,0.0,0-203-0-10904013001,2025,11
2,0-203-0-10904013001,E,2025-11-01T02:00:00Z,7.6,NaN,0.0,0-203-0-10904013001,2025,11
3,0-203-0-10904013001,E,2025-11-01T03:00:00Z,7.8,NaN,0.0,0-203-0-10904013001,2025,11
4,0-203-0-10904013001,E,2025-11-01T04:00:00Z,7.4,NaN,0.0,0-203-0-10904013001,2025,11


In [90]:
df_weather['FLAG'].isna().sum()

np.int64(129386)

In [91]:
df_weather['FLAG'].notna().sum()

np.int64(2160)

In [92]:
df_weather['FLAG'].unique()

array([nan,  5.,  8.,  2.,  7.,  6.,  3.,  1.,  0.,  4.])

In [93]:
df_weather['QUALITY'].unique()

array([0., 4., 3.])

In [94]:
### mapping variables names in weather data 
df_weather['ELEMENT_NAME'] = df_weather['ELEMENT'].map(chmi_vars_dict)

df_weather['WSI_NAME'] = df_weather['WSI'].map(wsi_dict)


In [95]:
df_weather

,STATION,ELEMENT,DT,VAL,FLAG,QUALITY,WSI,YEAR,MONTH,ELEMENT_NAME,WSI_NAME
0,0-203-0-10904013001,E,2025-11-01T00:00:00Z,8.0,NaN,0.0,0-203-0-10904013001,2025,11,Tlak páry,"Praha, Komořany"
1,0-203-0-10904013001,E,2025-11-01T01:00:00Z,7.7,NaN,0.0,0-203-0-10904013001,2025,11,Tlak páry,"Praha, Komořany"
2,0-203-0-10904013001,E,2025-11-01T02:00:00Z,7.6,NaN,0.0,0-203-0-10904013001,2025,11,Tlak páry,"Praha, Komořany"
3,0-203-0-10904013001,E,2025-11-01T03:00:00Z,7.8,NaN,0.0,0-203-0-10904013001,2025,11,Tlak páry,"Praha, Komořany"
4,0-203-0-10904013001,E,2025-11-01T04:00:00Z,7.4,NaN,0.0,0-203-0-10904013001,2025,11,Tlak páry,"Praha, Komořany"
...,...,...,...,...,...,...,...,...,...,...,...
131541,0-20000-0-11518,W2,2025-11-30T19:00:00Z,4.0,NaN,0.0,0-20000-0-11518,2025,11,Průběh počasí 2,"Praha, Ruzyně"
131542,0-20000-0-11518,W2,2025-11-30T20:00:00Z,4.0,NaN,0.0,0-20000-0-11518,2025,11,Průběh počasí 2,"Praha, Ruzyně"
131543,0-20000-0-11518,W2,2025-11-30T21:00:00Z,4.0,NaN,0.0,0-20000-0-11518,2025,11,Průběh počasí 2,"Praha, Ruzyně"
131544,0-20000-0-11518,W2,2025-11-30T22:00:00Z,4.0,NaN,0.0,0-20000-0-11518,2025,11,Průběh počasí 2,"Praha, Ruzyně"


In [96]:
### add metadata about stations

stations_meta = pd.read_csv(os.path.join(data_dir, "chmi_weather_stations_metadata.csv"))

In [97]:
stations_meta

,WSI,GH_ID,BEGIN_DATE,END_DATE,FULL_NAME,GEOGR1,GEOGR2,ELEVATION
0,0-20000-0-04030,ZIS04030,2015-01-01T00:00:00Z,3999-12-31T23:59:00Z,Reykjavik,-21.903905,64.127653,51.0
1,0-20000-0-11406,L3CHEB01,1863-10-01T00:00:00Z,1919-12-31T23:59:00Z,Cheb,12.362892,50.076212,458.0
2,0-20000-0-11406,L3CHEB01,1933-05-07T00:00:00Z,1938-04-30T23:59:00Z,Cheb,12.388900,50.073900,471.0
3,0-20000-0-11406,L3CHEB01,1943-06-01T00:00:00Z,1945-01-31T23:59:00Z,Cheb,12.388900,50.073900,471.0
4,0-20000-0-11406,L3CHEB01,1951-01-01T00:00:00Z,1960-12-31T23:59:00Z,Cheb,12.388900,50.073900,471.0
...,...,...,...,...,...,...,...,...
5534,0-203-0-42109005003,B4ZITK01,2015-09-16T00:00:00Z,3999-12-31T23:59:00Z,Žítková,17.867778,48.992500,705.0
5535,0-203-0-42109007001,B1PITI01,1944-10-01T00:00:00Z,1961-12-31T23:59:00Z,Pitín,17.890500,49.000900,342.0
5536,0-203-0-42109026001,B1LOPE01,1902-01-01T00:00:00Z,1902-12-31T23:59:00Z,Lopeník,17.792800,48.945800,672.0
5537,0-203-0-42109026001,B1LOPE01,1936-03-15T00:00:00Z,1946-06-30T23:59:00Z,Lopeník,17.792800,48.945800,672.0


In [98]:
stations_meta = stations_meta[stations_meta['WSI'].isin(wsi_dict)]

stations_meta["END_DATE_DT"] = pd.to_datetime(stations_meta["END_DATE"], utc=True, errors="coerce")

stations_meta = (
    stations_meta.sort_values("END_DATE_DT")
    .drop_duplicates(subset="WSI", keep="last")
)

now_utc = pd.Timestamp.now(tz="UTC")
df_chmi_stat = df_chmi_stat[
    (df_chmi_stat["END_DATE_DT"] >= now_utc)
]

In [99]:
stations_meta

,WSI,GH_ID,BEGIN_DATE,END_DATE,FULL_NAME,GEOGR1,GEOGR2,ELEVATION,END_DATE_DT
2214,0-203-0-11514,P1PKLE01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416923,50.086634,190.70,3999-12-31 23:59:00+00:00
75,0-20000-0-11567,P1PKBE01,2010-10-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Kbely",14.538056,50.123333,284.50,3999-12-31 23:59:00+00:00
1273,0-203-0-11101007001,L4PRAH01,2015-03-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Brdy",13.818380,49.658890,862.00,3999-12-31 23:59:00+00:00
1101,0-203-0-10904013001,P1PKOM01,2017-12-07T12:00:00Z,3999-12-31T23:59:00Z,"Praha, Komořany",14.406360,49.988600,213.00,3999-12-31 23:59:00+00:00
68,0-20000-0-11520,P1PLIB01,2018-05-17T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Libuš",14.446944,50.007778,302.04,3999-12-31 23:59:00+00:00
65,0-20000-0-11519,P1PKAR01,2024-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Karlov",14.427778,50.069167,260.50,3999-12-31 23:59:00+00:00
62,0-20000-0-11518,P1PRUZ01,2000-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Ruzyně",14.255556,50.100278,364.00,3999-12-31 23:59:00+00:00
1429,0-203-0-11201024001,P1PBRE01,2013-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Břevnov",14.352689,50.080843,355.00,3999-12-31 23:59:00+00:00
2217,0-203-0-11515,P1PKLM01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416436,50.086341,190.70,3999-12-31 23:59:00+00:00


In [100]:
stations_sel = stations_meta[["WSI", "FULL_NAME", "ELEVATION", "GEOGR1", "GEOGR2"]].copy()
stations_sel = stations_sel.rename(columns={
    "GEOGR1": "LON",
    "GEOGR2": "LAT"
})

df_weather["WSI"] = df_weather["WSI"].astype(str)
stations_sel["WSI"] = stations_sel["WSI"].astype(str)

In [101]:
df_weather_ext = df_weather.merge(stations_sel, on="WSI", how="left")


In [102]:
df_weather_ext

,STATION,ELEMENT,DT,VAL,FLAG,QUALITY,WSI,YEAR,MONTH,ELEMENT_NAME,WSI_NAME,FULL_NAME,ELEVATION,LON,LAT
0,0-203-0-10904013001,E,2025-11-01T00:00:00Z,8.0,NaN,0.0,0-203-0-10904013001,2025,11,Tlak páry,"Praha, Komořany","Praha, Komořany",213.0,14.406360,49.988600
1,0-203-0-10904013001,E,2025-11-01T01:00:00Z,7.7,NaN,0.0,0-203-0-10904013001,2025,11,Tlak páry,"Praha, Komořany","Praha, Komořany",213.0,14.406360,49.988600
2,0-203-0-10904013001,E,2025-11-01T02:00:00Z,7.6,NaN,0.0,0-203-0-10904013001,2025,11,Tlak páry,"Praha, Komořany","Praha, Komořany",213.0,14.406360,49.988600
3,0-203-0-10904013001,E,2025-11-01T03:00:00Z,7.8,NaN,0.0,0-203-0-10904013001,2025,11,Tlak páry,"Praha, Komořany","Praha, Komořany",213.0,14.406360,49.988600
4,0-203-0-10904013001,E,2025-11-01T04:00:00Z,7.4,NaN,0.0,0-203-0-10904013001,2025,11,Tlak páry,"Praha, Komořany","Praha, Komořany",213.0,14.406360,49.988600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131541,0-20000-0-11518,W2,2025-11-30T19:00:00Z,4.0,NaN,0.0,0-20000-0-11518,2025,11,Průběh počasí 2,"Praha, Ruzyně","Praha, Ruzyně",364.0,14.255556,50.100278
131542,0-20000-0-11518,W2,2025-11-30T20:00:00Z,4.0,NaN,0.0,0-20000-0-11518,2025,11,Průběh počasí 2,"Praha, Ruzyně","Praha, Ruzyně",364.0,14.255556,50.100278
131543,0-20000-0-11518,W2,2025-11-30T21:00:00Z,4.0,NaN,0.0,0-20000-0-11518,2025,11,Průběh počasí 2,"Praha, Ruzyně","Praha, Ruzyně",364.0,14.255556,50.100278
131544,0-20000-0-11518,W2,2025-11-30T22:00:00Z,4.0,NaN,0.0,0-20000-0-11518,2025,11,Průběh počasí 2,"Praha, Ruzyně","Praha, Ruzyně",364.0,14.255556,50.100278


In [103]:
print((df_weather_ext["FULL_NAME"] != df_weather_ext["WSI_NAME"]).any())
print((df_weather_ext["WSI"] != df_weather_ext["STATION"]).any())

False
False


In [116]:
### air quality chmi data

In [104]:
air_stations_meta = pd.read_csv(os.path.join(data_dir, "airquality_CHMI_stations_metadata.csv"))

In [105]:
air_stations_meta

,id_registration,station_code,locality_code,locality_name,street,city,lon,lat,alt,component_code,component_name,unit
0,40555,TOFFA,TOFF,Ostrava-Fifejdy,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,SO2,oxid siřičitý,ug/m^3
1,40557,TOFFA,TOFF,Ostrava-Fifejdy,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,NO2,oxid dusičitý,ug/m^3
2,40560,TOFFA,TOFF,Ostrava-Fifejdy,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,NOx,oxidy dusíku,ug/m^3
3,40559,TOFFA,TOFF,Ostrava-Fifejdy,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,O3,přízemní ozon,ug/m^3
4,40561,TOFFA,TOFF,Ostrava-Fifejdy,Gen. Janouška,Ostrava - Fifejdy,18.263689,49.839188,220 m,PM10,částice PM10,ug/m^3
...,...,...,...,...,...,...,...,...,...,...,...,...
479,1648406,TRYCA,TRYC,Rychvald,Mírová,Rychvald,18.377254,49.871670,241 m,INDX,Index kvality ovzduší,NaN
480,1410468,ACHOA,ACHO,Praha 4-Chodov,Vejvanovského,Praha 4,14.517450,50.030170,300 m,NO2,oxid dusičitý,ug/m^3
481,1410474,ACHOA,ACHO,Praha 4-Chodov,Vejvanovského,Praha 4,14.517450,50.030170,300 m,NOx,oxidy dusíku,ug/m^3
482,1410479,ACHOA,ACHO,Praha 4-Chodov,Vejvanovského,Praha 4,14.517450,50.030170,300 m,PM10,částice PM10,ug/m^3


In [106]:
air_stations_meta = air_stations_meta[air_stations_meta['locality_name'].str.contains('Praha', case=False, na=False)]


In [107]:
air_stations_meta

,id_registration,station_code,locality_code,locality_name,street,city,lon,lat,alt,component_code,component_name,unit
39,783575,AREPA,AREP,Praha 1-n. Republiky,NaN,NaN,14.429220,50.088066,190 m,NO2,oxid dusičitý,ug/m^3
40,783579,AREPA,AREP,Praha 1-n. Republiky,NaN,NaN,14.429220,50.088066,190 m,NOx,oxidy dusíku,ug/m^3
41,783591,AREPA,AREP,Praha 1-n. Republiky,NaN,NaN,14.429220,50.088066,190 m,PM10,částice PM10,ug/m^3
42,1648334,AREPA,AREP,Praha 1-n. Republiky,NaN,NaN,14.429220,50.088066,190 m,INDX,Index kvality ovzduší,NaN
66,41157,ALEGA,ALEG,Praha 2-Legerova (hot spot),Legerov 1843,Praha 2,14.430673,50.072388,219 m,NO2,oxid dusičitý,ug/m^3
...,...,...,...,...,...,...,...,...,...,...,...,...
436,1648433,ABREA,ABRE,Praha 6-Břevnov,Šlikova,Praha 6,14.380116,50.084385,300 m,INDX,Index kvality ovzduší,NaN
480,1410468,ACHOA,ACHO,Praha 4-Chodov,Vejvanovského,Praha 4,14.517450,50.030170,300 m,NO2,oxid dusičitý,ug/m^3
481,1410474,ACHOA,ACHO,Praha 4-Chodov,Vejvanovského,Praha 4,14.517450,50.030170,300 m,NOx,oxidy dusíku,ug/m^3
482,1410479,ACHOA,ACHO,Praha 4-Chodov,Vejvanovského,Praha 4,14.517450,50.030170,300 m,PM10,částice PM10,ug/m^3


In [ ]:
### this is only for one import file, i.e. one day

# df_air_qual = pd.read_csv(os.path.join(data_dir, "airquality_CHMI_stations_data.csv"))

In [116]:
df_air_qual = pd.read_csv(os.path.join(data_dir, "airquality_CHMI_1hour.csv"))

In [117]:
df_air_qual

,idRegistration,startTime,idValueType,value,source_file
0,10221,2025-11-01T00:00:00Z,8,1.3,airquality_1h_avg_CZ_2025110100.csv
1,40237,2025-11-01T00:00:00Z,8,48.7,airquality_1h_avg_CZ_2025110100.csv
2,40238,2025-11-01T00:00:00Z,8,6.2,airquality_1h_avg_CZ_2025110100.csv
3,40244,2025-11-01T00:00:00Z,8,12.6,airquality_1h_avg_CZ_2025110100.csv
4,40257,2025-11-01T00:00:00Z,6,-5009.0,airquality_1h_avg_CZ_2025110100.csv
...,...,...,...,...,...
345776,2053564,2025-11-30T23:00:00Z,6,-5009.0,airquality_1h_avg_CZ_2025113023.csv
345777,2140455,2025-11-30T23:00:00Z,6,-5009.0,airquality_1h_avg_CZ_2025113023.csv
345778,2140459,2025-11-30T23:00:00Z,6,-5009.0,airquality_1h_avg_CZ_2025113023.csv
345779,2140463,2025-11-30T23:00:00Z,6,-5009.0,airquality_1h_avg_CZ_2025113023.csv


In [118]:
### match with metadata based on idregistration

df_air_qual = df_air_qual.rename(columns={
    "idRegistration": "id_registration"
}
)

df_air_qual["id_registration"] = df_air_qual["id_registration"].astype(str)
air_stations_meta["id_registration"] = air_stations_meta["id_registration"].astype(str)

In [119]:
# join

df_air_qual = df_air_qual.merge(air_stations_meta, on="id_registration", how="right")

In [120]:
df_air_qual

,id_registration,startTime,idValueType,value,source_file,station_code,locality_code,locality_name,street,city,lon,lat,alt,component_code,component_name,unit
0,783575,2025-11-01T00:00:00Z,8,42.1,airquality_1h_avg_CZ_2025110100.csv,AREPA,AREP,Praha 1-n. Republiky,NaN,NaN,14.42922,50.088066,190 m,NO2,oxid dusičitý,ug/m^3
1,783575,2025-11-01T01:00:00Z,8,36.0,airquality_1h_avg_CZ_2025110101.csv,AREPA,AREP,Praha 1-n. Republiky,NaN,NaN,14.42922,50.088066,190 m,NO2,oxid dusičitý,ug/m^3
2,783575,2025-11-01T02:00:00Z,8,36.2,airquality_1h_avg_CZ_2025110102.csv,AREPA,AREP,Praha 1-n. Republiky,NaN,NaN,14.42922,50.088066,190 m,NO2,oxid dusičitý,ug/m^3
3,783575,2025-11-01T03:00:00Z,8,28.3,airquality_1h_avg_CZ_2025110103.csv,AREPA,AREP,Praha 1-n. Republiky,NaN,NaN,14.42922,50.088066,190 m,NO2,oxid dusičitý,ug/m^3
4,783575,2025-11-01T04:00:00Z,8,26.6,airquality_1h_avg_CZ_2025110104.csv,AREPA,AREP,Praha 1-n. Republiky,NaN,NaN,14.42922,50.088066,190 m,NO2,oxid dusičitý,ug/m^3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47417,1648733,2025-11-30T19:00:00Z,148,1.0,airquality_1h_avg_CZ_2025113019.csv,ACHOA,ACHO,Praha 4-Chodov,Vejvanovského,Praha 4,14.51745,50.030170,300 m,INDX,Index kvality ovzduší,NaN
47418,1648733,2025-11-30T20:00:00Z,148,1.0,airquality_1h_avg_CZ_2025113020.csv,ACHOA,ACHO,Praha 4-Chodov,Vejvanovského,Praha 4,14.51745,50.030170,300 m,INDX,Index kvality ovzduší,NaN
47419,1648733,2025-11-30T21:00:00Z,148,1.0,airquality_1h_avg_CZ_2025113021.csv,ACHOA,ACHO,Praha 4-Chodov,Vejvanovského,Praha 4,14.51745,50.030170,300 m,INDX,Index kvality ovzduší,NaN
47420,1648733,2025-11-30T22:00:00Z,148,1.0,airquality_1h_avg_CZ_2025113022.csv,ACHOA,ACHO,Praha 4-Chodov,Vejvanovského,Praha 4,14.51745,50.030170,300 m,INDX,Index kvality ovzduší,NaN


In [ ]:
### golemio air quality 

#### air quality metadata processing

# station_cols = {
#     'geometry.coordinates': 'coordinates', 
#     'properties.id': 'id', 
#     'properties.name': 'name', 
#     'properties.district': 'district', 
#     'properties.measurement.components.type': 'components'
# }

# air_quality_stations = df[station_cols.keys()]

# air_quality_stations = air_quality_stations.rename(columns=station_cols)

# air_quality_stations = (
#     air_quality_stations.groupby('id', as_index=False)
#     .agg({
#         'coordinates': 'first',
#         'name': 'first',
#         'district': 'first',
#         'components': list
#     })
# )

# air_quality_stations[["lon", "lat"]] = pd.DataFrame(air_quality_stations["coordinates"].tolist(), index=air_quality_stations.index)
# air_quality_stations["lon"] = pd.to_numeric(air_quality_stations["lon"], errors="coerce")
# air_quality_stations["lat"] = pd.to_numeric(air_quality_stations["lat"], errors="coerce")


In [ ]:

#air quality stations dictionary

# air_stat_dict = dict(
#     zip(
#         air_quality_stations['id'].astype(str),
#         air_quality_stations['name'].astype(str)  
#     )
# )

In [121]:
### loading the dictionaries


def load_wsi_dict(path="data/raw/wsi_dict.csv"):
    df = pd.read_csv(path, dtype=str)
    return dict(df.values)

def load_chmi_vars(path="data/raw/chmi_vars.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [122]:
# list unique measured weather variables
df_weather_ext[["ELEMENT", "ELEMENT_NAME"]].drop_duplicates().sort_values("ELEMENT")

# ?keep: D, F, H, P, SRA10M, T

,ELEMENT,ELEMENT_NAME
1440,C-C1Av,Ceilometr CLOUD_1 AVG
2160,C-C1Co,Ceilometr CLOUD_1 COUNT
2880,C-C1Mn,Ceilometr CLOUD_1 MIN
3600,C-C1Mx,Ceilometr CLOUD_1 MAX
4320,C-C2Av,Ceilometr CLOUD_2 AVG
5040,C-C2Co,Ceilometr CLOUD_2 COUNT
5760,C-C2Mn,Ceilometr CLOUD_2 MIN
6480,C-C2Mx,Ceilometr CLOUD_2 MAX
7200,C-C3Av,Ceilometr CLOUD_3 AVG
7920,C-C3Co,Ceilometr CLOUD_3 COUNT


In [123]:
# list all unique pollutants
df_air_qual[["component_code", "component_name", "unit"]].drop_duplicates()

,component_code,component_name,unit
0,NO2,oxid dusičitý,ug/m^3
719,NOx,oxidy dusíku,ug/m^3
1438,PM10,částice PM10,ug/m^3
2157,INDX,Index kvality ovzduší,NaN
4304,CO,oxid uhelnatý,ug/m^3
5018,PM2_5,"jemné částice PM2,5",ug/m^3
11479,O3,přízemní ozon,ug/m^3
13636,SO2,oxid siřičitý,ug/m^3


### Connect stations by their distance

In [124]:
# install if needed
#pip install geopandas folium mapclassify

In [126]:
stations_meta

,WSI,GH_ID,BEGIN_DATE,END_DATE,FULL_NAME,GEOGR1,GEOGR2,ELEVATION,END_DATE_DT
2214,0-203-0-11514,P1PKLE01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416923,50.086634,190.70,3999-12-31 23:59:00+00:00
75,0-20000-0-11567,P1PKBE01,2010-10-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Kbely",14.538056,50.123333,284.50,3999-12-31 23:59:00+00:00
1273,0-203-0-11101007001,L4PRAH01,2015-03-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Brdy",13.818380,49.658890,862.00,3999-12-31 23:59:00+00:00
1101,0-203-0-10904013001,P1PKOM01,2017-12-07T12:00:00Z,3999-12-31T23:59:00Z,"Praha, Komořany",14.406360,49.988600,213.00,3999-12-31 23:59:00+00:00
68,0-20000-0-11520,P1PLIB01,2018-05-17T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Libuš",14.446944,50.007778,302.04,3999-12-31 23:59:00+00:00
65,0-20000-0-11519,P1PKAR01,2024-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Karlov",14.427778,50.069167,260.50,3999-12-31 23:59:00+00:00
62,0-20000-0-11518,P1PRUZ01,2000-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Ruzyně",14.255556,50.100278,364.00,3999-12-31 23:59:00+00:00
1429,0-203-0-11201024001,P1PBRE01,2013-09-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Břevnov",14.352689,50.080843,355.00,3999-12-31 23:59:00+00:00
2217,0-203-0-11515,P1PKLM01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416436,50.086341,190.70,3999-12-31 23:59:00+00:00


In [ ]:
# list all weather and air quality stations ----
weather_stations = (
    stations_meta
    .loc[:, ["WSI", "FULL_NAME", "GEOGR1", "GEOGR2", "ELEVATION"]]
    .drop_duplicates()
    .rename(columns={
        "WSI": "weather_station_id",
        "FULL_NAME": "weather_station_name",
        "GEOGR1": "weather_lon",
        "GEOGR2": "weather_lat",
        "ELEVATION": "weather_alt"
    })
)

air_stations = (
    air_stations_meta
    .loc[
        air_stations_meta["locality_name"].str.contains("Praha", case=False, na=False),
        ["station_code", "locality_name", "lon", "lat", "alt"]
    ]
    .drop_duplicates()
    .rename(columns={
        "station_code": "air_station_code",
        "locality_name": "air_station_name",
        "lon": "air_lon",
        "lat": "air_lat",
        "alt": "air_alt"
    })
)


In [128]:
weather_stations

,weather_station_id,weather_station_name,weather_lon,weather_lat,weather_alt
2214,0-203-0-11514,"Praha, Klementinum",14.416923,50.086634,190.70
75,0-20000-0-11567,"Praha, Kbely",14.538056,50.123333,284.50
1273,0-203-0-11101007001,"Praha, Brdy",13.818380,49.658890,862.00
1101,0-203-0-10904013001,"Praha, Komořany",14.406360,49.988600,213.00
68,0-20000-0-11520,"Praha, Libuš",14.446944,50.007778,302.04
65,0-20000-0-11519,"Praha, Karlov",14.427778,50.069167,260.50
62,0-20000-0-11518,"Praha, Ruzyně",14.255556,50.100278,364.00
1429,0-203-0-11201024001,"Praha, Břevnov",14.352689,50.080843,355.00
2217,0-203-0-11515,"Praha, Klementinum",14.416436,50.086341,190.70


In [129]:
air_stations

,air_station_code,air_station_name,air_lon,air_lat,air_alt
39,AREPA,Praha 1-n. Republiky,14.429220,50.088066,190 m
66,ALEGA,Praha 2-Legerova (hot spot),14.430673,50.072388,219 m
72,AKALA,Praha 8-Karlín,14.442049,50.094238,203 m
76,AVYNA,Praha 9-Vysočany,14.503096,50.111080,219 m
93,ALIBA,Praha 4-Libuš,14.445933,50.007305,301 m
215,ASTOA,Praha 5-Stodůlky,14.331413,50.046131,309 m
219,APRUA,Praha 10-Průmyslová,14.537820,50.062298,267 m
248,AHOLA,Praha 7-Holešovice,14.443650,50.108845,185 m
279,AKOBA,Praha 8-Kobylisy,14.467578,50.122189,269 m
315,ASUCA,Praha 6-Suchdol,14.384639,50.126530,277 m


In [ ]:
# match stations ----
# convert both station lists to geopandas dataframes
air_gdf = gpd.GeoDataFrame(
    air_stations,
    geometry=gpd.points_from_xy(
        air_stations["air_lon"],
        air_stations["air_lat"]
    ),
    crs="EPSG:4326"
)
weather_gdf = gpd.GeoDataFrame(
    weather_stations,
    geometry=gpd.points_from_xy(
        weather_stations["weather_lon"],
        weather_stations["weather_lat"]
    ),
    crs="EPSG:4326"
)
# convert to meters
air_gdf_m = air_gdf.to_crs("EPSG:5514")
weather_gdf_m = weather_gdf.to_crs("EPSG:5514")
# find nearest matches
nearest_match = gpd.sjoin_nearest(
    air_gdf_m,
    weather_gdf_m,
    how="left",
    distance_col="distance_m"
)
# add column for km
nearest_match["distance_km"] = nearest_match["distance_m"] / 1000
# clean cols
nearest_match = nearest_match[
    [
        "air_station_code",
        "air_station_name",
        "air_lon",
        "air_lat",
        "air_alt",
        "weather_station_id",
        "weather_station_name",
        "weather_lon",
        "weather_lat",
        "weather_alt",
        "distance_m",
        "distance_km"
    ]
].copy()
# print the matches by decreasing distance
nearest_match.sort_values("distance_km", ascending=False)[
    [
        "air_station_name",
        "weather_station_name",
        "distance_km"
    ]
]

,air_station_name,weather_station_name,distance_km
219,Praha 10-Průmyslová,"Praha, Kbely",6.788402
480,Praha 4-Chodov,"Praha, Libuš",5.632987
279,Praha 8-Kobylisy,"Praha, Kbely",5.041256
315,Praha 6-Suchdol,"Praha, Klementinum",5.002194
215,Praha 5-Stodůlky,"Praha, Břevnov",4.150278
248,Praha 7-Holešovice,"Praha, Klementinum",3.123899
76,Praha 9-Vysočany,"Praha, Kbely",2.847472
433,Praha 6-Břevnov,"Praha, Břevnov",2.002016
72,Praha 8-Karlín,"Praha, Klementinum",1.986876
413,Praha 2-Riegrovy sady,"Praha, Karlov",1.736525


In [ ]:
# Preprocess data before merging ----

# copies
air = df_air_qual.copy()
weather = df_weather_ext.copy()
station_match = nearest_match.copy()
# clean column names
air.columns = air.columns.str.strip()
weather.columns = weather.columns.str.strip()
station_match.columns = station_match.columns.str.strip()
# set time columns
air_time_col = "startTime"
weather_time_col = "DT"
# clean station names
air["locality_name"] = air["locality_name"].astype(str).str.strip()
weather["WSI_NAME"] = weather["WSI_NAME"].astype(str).str.strip()
station_match["air_station_name"] = station_match["air_station_name"].astype(str).str.strip()
station_match["weather_station_name"] = station_match["weather_station_name"].astype(str).str.strip()
# convert times
air[air_time_col] = pd.to_datetime(air[air_time_col]).dt.tz_localize(None)
weather[weather_time_col] = pd.to_datetime(weather[weather_time_col]).dt.tz_localize(None)
## keep one nearest weather station per air station
#station_match = (
 #   station_match
 #   .sort_values("distance_m")
  #  .drop_duplicates(subset=["air_station_name"], keep="first")
#)


In [ ]:
# Pivot wider air ----
air_wide = (
    air
    .pivot_table(
        index=[air_time_col, "locality_name"],
        columns="component_code",
        values="value",
        aggfunc="mean"
    )
    .reset_index()
)
air_wide.columns.name = None
air_wide = air_wide.rename(
    columns={
        col: f"air_{col}"
        for col in air_wide.columns
        if col not in [air_time_col, "locality_name"]
    }
)

air_wide.head()


,startTime,locality_name,air_CO,air_INDX,air_NO2,air_NOx,air_O3,air_PM10,air_PM2_5,air_SO2
0,2025-11-01,Praha 1-n. Republiky,NaN,3.0,42.1,158.6,NaN,45.4,NaN,NaN
1,2025-11-01,Praha 10-Průmyslová,NaN,2.0,30.4,96.2,NaN,32.5,NaN,NaN
2,2025-11-01,Praha 10-Vršovice,NaN,3.0,32.7,201.4,NaN,54.9,NaN,NaN
3,2025-11-01,Praha 2-Legerova (hot spot),1051.0,3.0,42.3,191.7,NaN,49.2,40.6,NaN
4,2025-11-01,Praha 2-Riegrovy sady,NaN,3.0,-5009.0,-5009.0,-5009.0,-5009.0,-5009.0,-5009.0


In [ ]:
# pivot wider weather ----
weather_wide = (
    weather
    .pivot_table(
        index=[weather_time_col, "WSI_NAME"],
        columns="ELEMENT",
        values="VAL",
        aggfunc="mean"
    )
    .reset_index()
)
weather_wide.columns.name = None
weather_wide = weather_wide.rename(
    columns={
        col: f"weather_{col}"
        for col in weather_wide.columns
        if col not in [weather_time_col, "WSI_NAME"]
    }
)

weather_wide.head()

,DT,WSI_NAME,weather_C-C1Av,weather_C-C1Co,weather_C-C1Mn,weather_C-C1Mx,weather_C-C2Av,weather_C-C2Co,weather_C-C2Mn,weather_C-C2Mx,...,weather_P_hm,weather_RGLB1H,weather_SRA1H,weather_SSV1H,weather_Td,weather_VV,weather_W1,weather_W2,weather_ppp,weather_ww
0,2025-11-01,"Praha, Břevnov",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-11-01,"Praha, Karlov",6719.0,225.0,5440.0,9270.0,7174.0,58.0,5720.0,9360.0,...,1019.3,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-11-01,"Praha, Kbely",6170.0,224.0,5440.0,6990.0,6385.0,26.0,5690.0,6880.0,...,1019.5,NaN,0.0,NaN,3.7,30.0,10.0,10.0,-0.1,508.0
3,2025-11-01,"Praha, Klementinum",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1018.7,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-11-01,"Praha, Komořany",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# prepare location matching ----
air_wide_with_match = air_wide.merge(
    station_match,
    left_on="locality_name",
    right_on="air_station_name",
    how="left",
    validate="many_to_one"
)

In [ ]:
# merge ----
merged_wide = air_wide_with_match.merge(
    weather_wide,
    left_on=[air_time_col, "weather_station_name"],
    right_on=[weather_time_col, "WSI_NAME"],
    how="left",
    validate="many_to_one"
)

print("Difference:", len(merged_wide) - len(air_wide)) # should be 0
merged_wide


Difference: 0


,startTime,locality_name,air_CO,air_INDX,air_NO2,air_NOx,air_O3,air_PM10,air_PM2_5,air_SO2,...,weather_P_hm,weather_RGLB1H,weather_SRA1H,weather_SSV1H,weather_Td,weather_VV,weather_W1,weather_W2,weather_ppp,weather_ww
0,2025-11-01 00:00:00,Praha 1-n. Republiky,NaN,3.0,42.1,158.6,NaN,45.4,NaN,NaN,...,1018.7,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-11-01 00:00:00,Praha 10-Průmyslová,NaN,2.0,30.4,96.2,NaN,32.5,NaN,NaN,...,1019.5,NaN,0.0,NaN,3.7,30.0,10.0,10.0,-0.1,508.0
2,2025-11-01 00:00:00,Praha 10-Vršovice,NaN,3.0,32.7,201.4,NaN,54.9,NaN,NaN,...,1019.3,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-11-01 00:00:00,Praha 2-Legerova (hot spot),1051.0,3.0,42.3,191.7,NaN,49.2,40.6,NaN,...,1019.3,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-11-01 00:00:00,Praha 2-Riegrovy sady,NaN,3.0,-5009.0,-5009.0,-5009.0,-5009.0,-5009.0,-5009.0,...,1019.3,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10061,2025-11-30 23:00:00,Praha 6-Suchdol,NaN,1.0,NaN,NaN,-5009.0,-5009.0,NaN,NaN,...,1016.8,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10062,2025-11-30 23:00:00,Praha 7-Holešovice,NaN,NaN,-5009.0,-5009.0,NaN,-5009.0,-5009.0,NaN,...,1016.8,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10063,2025-11-30 23:00:00,Praha 8-Karlín,NaN,2.0,-5009.0,-5009.0,NaN,-5009.0,NaN,NaN,...,1016.8,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10064,2025-11-30 23:00:00,Praha 8-Kobylisy,NaN,2.0,-5009.0,-5009.0,-5009.0,-5009.0,NaN,NaN,...,1018.1,NaN,0.0,NaN,0.9,5.0,6.0,4.0,0.9,61.0
